# Tres cuerpos, caos y N cuerpos gravitatorios

_Física Computacional 2026_

> Notebook generado a partir de la clase `04-1-n-cuerpos.qmd`. Ejecuten las celdas en orden.

> **Pregunta de la clase.** ¿Durante cuánto tiempo es predictiva la simulación
> de un sistema gravitatorio caótico, y qué podemos afirmar después de ese
> tiempo?

En la clase anterior el problema de dos cuerpos se reducía a uno, y la
cuadratura de la órbita nos daba una referencia independiente de la
trayectoria. Con tres cuerpos esa reducción desaparece: no hay fórmula cerrada
general y, en muchos casos, dos condiciones iniciales casi iguales terminan en
trayectorias completamente distintas. Al terminar podremos medir cuánto tiempo
vale una trayectoria calculada, separar el error numérico de la sensibilidad
física, y decir qué sigue siendo confiable después.

In [ ]:

import time

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation
from scipy.integrate import solve_ivp


def animar(figura, actualizar, cuadros, intervalo=60):
    """Animación de matplotlib embebida en la página, sin depender de ffmpeg."""
    pelicula = animation.FuncAnimation(figura, actualizar, frames=cuadros, interval=intervalo)
    html = pelicula.to_jshtml(default_mode="loop")
    plt.close(figura)
    return HTML(html)

## 1. De dos a tres cuerpos

Cada cuerpo $i$, de masa $m_i$ y posición $\mathbf r_i$, siente la atracción de
todos los demás:

$$
\ddot{\mathbf r}_i=G\sum_{j\neq i}m_j\,\frac{\mathbf r_j-\mathbf r_i}{|\mathbf r_j-\mathbf r_i|^3}.
$$

Usamos unidades con $G=1$. Se conservan la energía, el momento lineal y el
momento angular totales,

$$
E=\sum_i\frac12m_iv_i^2-\sum_{i<j}\frac{m_im_j}{r_{ij}},\qquad
\mathbf P=\sum_im_i\mathbf v_i,\qquad
\mathbf L=\sum_im_i\,\mathbf r_i\times\mathbf v_i ,
$$

y con $\mathbf P=0$ el centro de masa queda quieto. Con dos cuerpos estas
cantidades alcanzaban para reducir el problema a una cuadratura. Con tres
cuerpos en el plano hay doce ecuaciones de primer orden y no alcanzan: Bruns y
Poincaré mostraron a fines del siglo XIX que no hay suficientes cantidades
conservadas para una solución cerrada general.

Escribimos la aceleración para un número cualquiera de cuerpos, sin bucles en
Python: la diferencia $\mathbf r_j-\mathbf r_i$ de todas las parejas es un
arreglo de forma `(N, N, dimensión)`.

In [ ]:

def aceleraciones(r, m, suavizado=0.0):
    """Aceleración gravitatoria de cada cuerpo, con G = 1.

    r tiene forma (N, dimensión) y m forma (N,). Con suavizado ε > 0, la
    distancia r_ij se reemplaza por sqrt(r_ij² + ε²).
    """
    d = r[None, :, :] - r[:, None, :]  # d[i, j] = r_j − r_i
    d2 = np.sum(d**2, axis=-1) + suavizado**2
    np.fill_diagonal(d2, np.inf)  # sin autointeracción
    return np.einsum("ij,ijk->ik", m[None, :] * d2**-1.5, d)


def energia(r, v, m):
    """Energía total; r y v tienen forma (N, dimensión)."""
    cinetica = 0.5 * np.sum(m * np.sum(v**2, axis=-1))
    distancia = np.linalg.norm(r[None, :, :] - r[:, None, :], axis=-1)
    i, j = np.triu_indices(len(m), k=1)
    return cinetica - np.sum(m[i] * m[j] / distancia[i, j])


def momentos(r, v, m):
    """Momento lineal y momento angular total, para movimiento en el plano."""
    P = np.sum(m[:, None] * v, axis=0)
    L = np.sum(m * (r[:, 0] * v[:, 1] - r[:, 1] * v[:, 0]))
    return P, L


def integrar(y0, m, t_final, rtol=1e-12, t_eval=None):
    """Integra con DOP853. El estado es [posiciones, velocidades] en el plano."""
    n = len(m)

    def derivada(t, y):
        r = y[: 2 * n].reshape(n, 2)
        return np.concatenate([y[2 * n :], aceleraciones(r, m).ravel()])

    return solve_ivp(derivada, (0, t_final), y0, method="DOP853",
                     rtol=rtol, atol=rtol * 1e-2, t_eval=t_eval)


def separar(y, n):
    """Devuelve posiciones y velocidades con forma (n, 2) a partir del estado."""
    return y[: 2 * n].reshape(n, 2), y[2 * n :].reshape(n, 2)

`np.fill_diagonal(d2, np.inf)` elimina la autointeracción: la división por
infinito da cero en lugar de cero sobre cero.

## 2. La órbita en ocho como test

Sin solución general, ¿contra qué comparamos? Contra **soluciones especiales**.
En 2000, [Chenciner y Montgomery](https://doi.org/10.2307/2661357) demostraron
que existe una órbita periódica en la que tres masas iguales se persiguen sobre
una curva en forma de ocho. Las condiciones iniciales, calculadas numéricamente
por [Moore](https://doi.org/10.1103/PhysRevLett.70.3675) y
[Simó](https://doi.org/10.1007/978-3-0348-8268-2_6), se conocen con ocho cifras.

In [ ]:

m_ocho = np.ones(3)
x1 = np.array([0.97000436, -0.24308753])
v3 = np.array([-0.93240737, -0.86473146])
y_ocho = np.concatenate([x1, -x1, [0.0, 0.0], -v3 / 2, -v3 / 2, v3])
periodo = 6.32591398

ocho = integrar(y_ocho, m_ocho, periodo, t_eval=np.linspace(0, periodo, 400))

In [ ]:

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(ocho.y[0], ocho.y[1], color="0.85", lw=1)
cuerpos = [ax.plot([], [], "o", ms=8, color=f"C{i}")[0] for i in range(3)]
estelas = [ax.plot([], [], lw=1.5, color=f"C{i}")[0] for i in range(3)]
ax.set(xlim=(-1.2, 1.2), ylim=(-0.5, 0.5), aspect="equal", xticks=[], yticks=[])


def cuadro(k):
    k = 4 * k
    for i in range(3):
        cuerpos[i].set_data([ocho.y[2 * i, k]], [ocho.y[2 * i + 1, k]])
        estelas[i].set_data(ocho.y[2 * i, max(0, k - 40) : k + 1], ocho.y[2 * i + 1, max(0, k - 40) : k + 1])
    return cuerpos + estelas


animar(fig, cuadro, cuadros=100)

In [ ]:

fig, ax = plt.subplots(figsize=(7, 3.5))
for i in range(3):
    ax.plot(ocho.y[2 * i], ocho.y[2 * i + 1], lw=1)
    ax.plot(*y_ocho[2 * i : 2 * i + 2], "o", ms=6, color=f"C{i}")
ax.set(xlabel="$x$", ylabel="$y$", aspect="equal")
ax.grid(alpha=0.25)
plt.show()

Un test natural: después de un período el estado tiene que volver al inicial.

**Antes de ejecutar:** si pasamos de `rtol = 1e-10` a `rtol = 1e-12`, ¿cuánto
esperan que mejore la distancia entre el estado final y el inicial?

In [ ]:

for rtol in (1e-10, 1e-12):
    sol = integrar(y_ocho, m_ocho, periodo, rtol=rtol)
    r, v = separar(sol.y[:, -1], 3)
    r0, v0 = separar(y_ocho, 3)
    print(f"rtol = {rtol:.0e}: |y(T) − y(0)| = {np.linalg.norm(sol.y[:, -1] - y_ocho):.1e}, "
          f"|ΔE/E| = {abs(energia(r, v, m_ocho) / energia(r0, v0, m_ocho) - 1):.1e}")

Un test tiene su propio límite de resolución. Antes de exigirle una tolerancia,
hay que preguntarse qué la limita: el integrador, o los datos con que se armó.

## 3. Encuentros cercanos: el problema pitagórico

En 1913, [Burrau](https://doi.org/10.1002/asna.19131950602) propuso un problema de apariencia inocente: tres masas $3$, $4$
y $5$ en reposo en los vértices de un triángulo rectángulo de lados $3$, $4$ y
$5$, cada una opuesta al lado de su misma longitud. El centro de masa está en
el origen y el momento angular es nulo.

In [ ]:

m_burrau = np.array([3.0, 4.0, 5.0])
y_burrau = np.array([1, 3, -2, -1, 1, -1, 0, 0, 0, 0, 0, 0], dtype=float)
t_final = 70.0

burrau = integrar(y_burrau, m_burrau, t_final)
r0, v0 = separar(y_burrau, 3)
rf, vf = separar(burrau.y[:, -1], 3)
print(f"pasos: {len(burrau.t) - 1}, evaluaciones: {burrau.nfev}, "
      f"paso máximo / mínimo: {np.diff(burrau.t).max() / np.diff(burrau.t).min():.0e}")
print(f"|ΔE/E| al final: {abs(energia(rf, vf, m_burrau) / energia(r0, v0, m_burrau) - 1):.1e}")
P0, L0 = momentos(r0, v0, m_burrau)
Pf, Lf = momentos(rf, vf, m_burrau)
print(f"|ΔP| al final: {np.linalg.norm(Pf - P0):.1e},  |ΔL| al final: {abs(Lf - L0):.1e}")

Para la animación integramos otra vez en tiempos equiespaciados, y una segunda
vez con la coordenada $x$ de la masa $3$ corrida en $10^{-8}$.

In [ ]:

t_anim = np.linspace(0, t_final, 1401)
y_vecina = y_burrau.copy()
y_vecina[0] += 1e-8
burrau_a = integrar(y_burrau, m_burrau, t_final, rtol=1e-13, t_eval=t_anim)
burrau_b = integrar(y_vecina, m_burrau, t_final, rtol=1e-13, t_eval=t_anim)

En color, los tres cuerpos; en negro, los de la integración corrida. Por ahora
miren cómo se mueven los cuerpos: a la segunda trayectoria volvemos en la
sección 4.

In [ ]:

a, b = burrau_a.y, burrau_b.y
fig, ax = plt.subplots(figsize=(5, 5))
cuerpos = [ax.plot([], [], "o", ms=4 + m, color=f"C{i}")[0] for i, m in enumerate(m_burrau)]
estelas = [ax.plot([], [], lw=1, color=f"C{i}")[0] for i in range(3)]
vecina = [ax.plot([], [], "o", ms=4 + m, mfc="none", mec="black")[0] for m in m_burrau]
reloj = ax.text(0.03, 0.95, "", transform=ax.transAxes)
ax.set(xlim=(-7, 7), ylim=(-7, 7), aspect="equal", xticks=[], yticks=[])


def cuadro(k):
    k = 10 * k
    for i in range(3):
        cuerpos[i].set_data([a[2 * i, k]], [a[2 * i + 1, k]])
        estelas[i].set_data(a[2 * i, max(0, k - 60) : k + 1], a[2 * i + 1, max(0, k - 60) : k + 1])
        vecina[i].set_data([b[2 * i, k]], [b[2 * i + 1, k]])
    reloj.set_text(f"t = {t_anim[k]:4.1f}")
    return cuerpos + estelas + vecina + [reloj]


animar(fig, cuadro, cuadros=141)

In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
for i in range(3):
    ax.plot(burrau.y[2 * i], burrau.y[2 * i + 1], lw=0.6, label=f"m = {m_burrau[i]:g}")
    ax.plot(*y_burrau[2 * i : 2 * i + 2], "o", color=f"C{i}")
ax.set(xlabel="$x$", ylabel="$y$", aspect="equal", xlim=(-6, 6), ylim=(-6, 6))
ax.legend()
ax.grid(alpha=0.25)
plt.show()

Hay encuentros cercanos: dos cuerpos pasan a distancias mucho menores que el
tamaño del sistema. Cerca de un encuentro la aceleración crece como $1/r^2$ y
la escala de tiempo se acorta como $r^{3/2}$.

In [ ]:

parejas = ((0, 1), (0, 2), (1, 2))
posiciones = burrau.y[:6].reshape(3, 2, -1)
distancias = np.array([np.linalg.norm(posiciones[i] - posiciones[j], axis=0) for i, j in parejas])
paso = np.diff(burrau.t)

In [ ]:

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(burrau.t[:-1], paso, lw=0.8, label="paso $\\Delta t$")
ax.semilogy(burrau.t, distancias.min(axis=0), lw=0.8, label="distancia mínima")
ax.set(xlabel="$t$")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

El paso adaptativo se achica cerca de las distancias mínimas. Un método de paso
fijo elegido para resolver el encuentro más cercano tendría que usar ese paso
chico durante toda la integración: en lugar de algunos miles de pasos serían
cientos de millones.

### 3.1. ¿Y la regularización?

Acá **no regularizamos**: integramos las ecuaciones de Newton tal cual, sin
suavizado, y dejamos que el paso adaptativo se achique en cada encuentro.

In [ ]:

i, k = np.unravel_index(np.argmin(distancias), distancias.shape)
print(f"encuentro más cercano: masas {m_burrau[parejas[i][0]]:g} y {m_burrau[parejas[i][1]]:g}, "
      f"a distancia {distancias[i, k]:.1e} en t = {burrau.t[k]:.2f}")
print(f"pasos con Δt < 1e-4: {np.sum(paso < 1e-4)} de {len(paso)}")

Para esta corrida, el encuentro más cercano es del orden de $10^{-4}$ del
tamaño del sistema y los tres invariantes cambian poco. Es un control necesario
contra un error numérico grueso, pero no demuestra que la trayectoria detallada
sea correcta: esa limitación aparece enseguida por el caos. El cálculo tiene dos
precios:

- **costo**: una parte importante de los pasos se gasta en unos pocos
  encuentros;
- **redondeo**: la distancia $\mathbf r_i-\mathbf r_j$ es la diferencia de dos
  números de orden uno. Con $r_{ij}\sim10^{-4}$ se pierden cuatro de las
  dieciséis cifras antes de empezar.

En un encuentro todavía más cercano, o en una colisión exacta, la fuerza
$1/r^2$ diverge y ningún paso en el tiempo físico alcanza. Para el problema
relativo de dos cuerpos, la **regularización** es un cambio de variables que
elimina esa singularidad:

1. **Cambio de tiempo** ([Sundman](https://doi.org/10.1007/BF02422379)): se
   integra en un tiempo ficticio $s$ con $dt=r\,ds$, donde $r$ es la distancia
   entre los dos cuerpos. Un paso uniforme en $s$ es un paso en $t$
   proporcional a la distancia: lo que hace el paso adaptativo, pero
   incorporado a las ecuaciones.
2. **Cambio de coordenadas** ([Levi-Civita](https://doi.org/10.1007/BF02404404)):
   en el plano, escribimos la posición relativa $\mathbf r_2-\mathbf r_1$ como
   un número complejo $z=x+iy$. La ecuación de movimiento relativo es
   $\ddot z=-\mu\,z/|z|^3$, con $\mu=m_1+m_2$, y su energía por unidad de masa
   reducida, $h=\tfrac12|\dot z|^2-\mu/|z|$, se conserva. Definimos una nueva
   variable compleja $w$ por $z=w^2$. Con el tiempo $s$ del punto anterior, la
   ecuación de movimiento queda

   $$
   \frac{d^2w}{ds^2}=\frac h2\,w .
   $$

   Para una órbita ligada $h<0$, y esta es la ecuación de un **oscilador
   armónico** de frecuencia $\sqrt{-h/2}$, sin ninguna singularidad: la
   colisión $z=0$ es simplemente el paso de $w$ por el origen.

En tres dimensiones los números complejos no alcanzan. La transformación de
[Kustaanheimo y Stiefel](https://doi.org/10.1515/crll.1965.218.204) usa cuatro
variables en lugar de tres y lleva el problema de Kepler a un oscilador
armónico en cuatro dimensiones.

Con las ecuaciones regularizadas se puede integrar **a través** de una
colisión. En un sistema de tres o más cuerpos se regularizan los pares cercanos
y se incluyen las perturbaciones de los demás; así resolvieron
[Szebehely y Peters](https://doi.org/10.1086/110355) el problema pitagórico en
1967. Los códigos de muchos cuerpos de
producción también tratan de modo especial los binarios cercanos.

**Para pensar.** El oscilador armónico es el sistema mejor conocido de la
clase 02. ¿Qué ventaja tiene integrar un encuentro cercano con Verlet en la
variable $w$ en lugar de hacerlo en $z$?

## 4. Caos y horizonte de predicción

### 4.1. Dos trayectorias vecinas

Integramos dos veces el mismo sistema, con la coordenada $x$ del primer cuerpo
corrida en $\delta_0=10^{-8}$, y medimos la distancia entre los dos estados en el
espacio de fases,

$$
\delta(t)=|\,y_a(t)-y_b(t)\,| .
$$

Lo hacemos con tres sistemas: el problema pitagórico, la órbita en ocho y dos
cuerpos iguales en una órbita de Kepler con excentricidad $0.5$.

**Antes de ejecutar:** ¿cómo esperan que crezca $\delta(t)$ en cada caso? Para
Kepler, piensen qué cambia en la órbita si la energía cambia un poco.

In [ ]:

def vecinas(y0, m, delta0, t_final, componente=0, rtol=1e-13):
    """Dos integraciones que empiezan a distancia delta0 en una componente del estado."""
    t = np.linspace(0, t_final, 1401)
    y_b = y0.copy()
    y_b[componente] += delta0
    a = integrar(y0, m, t_final, rtol, t_eval=t)
    b = integrar(y_b, m, t_final, rtol, t_eval=t)
    return t, a.y, b.y


# Kepler: dos masas iguales, semieje relativo 1, excentricidad 0.5, en el pericentro
m_kepler = np.ones(2)
e, masa_total = 0.5, 2.0
r_p, v_p = 1 - e, np.sqrt(masa_total * (1 + e) / (1 - e))
y_kepler = np.array([r_p / 2, 0, -r_p / 2, 0, 0, v_p / 2, 0, -v_p / 2])

sistemas = {
    "Kepler": (y_kepler, m_kepler),
    "ocho": (y_ocho, m_ocho),
    "pitagórico": (y_burrau, m_burrau),
}
trayectorias = {nombre: vecinas(y0, m, 1e-8, t_final) for nombre, (y0, m) in sistemas.items()}
separaciones = {nombre: (t, np.linalg.norm(a - b, axis=0)) for nombre, (t, a, b) in trayectorias.items()}

In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
for nombre, (t, delta) in separaciones.items():
    ax.semilogy(t, delta, lw=1, label=nombre)
ax.set(xlabel="$t$", ylabel="$\\delta(t)$")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

Tres comportamientos distintos:

- **Kepler.** La perturbación cambia un poco la energía, y con ella el período.
  Las dos órbitas se desfasan a velocidad constante: $\delta$ crece en forma
  **lineal** con $t$. En escala logarítmica se ve como una curva que se aplana.
  Los picos en cada pericentro no vienen de la integración: el desfasaje
  temporal $\Delta t$ es el mismo, pero ahí el estado cambia más rápido, y
  $\delta\approx|\dot y|\,\Delta t$.
- **Ocho.** La órbita es estable: la perturbación crece poco.
- **Pitagórico.** Hay cuatro etapas:
  - hasta $t\approx32$, $\delta$ sube y baja de a saltos, en los encuentros
    cercanos;
  - entre $t\approx32$ y $47$, con los tres cuerpos interactuando, crece en
    forma **exponencial**: una recta en escala logarítmica;
  - entre $t\approx47$ y $58$, las masas $4$ y $5$ forman un binario y la masa
    $3$ se aleja sin escapar: $\delta$ crece mucho más lento;
  - en $t\approx60$, un último encuentro entre $4$ y $5$ expulsa a la masa $3$,
    y $\delta$ da un salto final hasta el orden de uno.

Los tres cuerpos arrancan en reposo, pero la energía total no es cero: la
cinética es nula y la potencial es negativa,

$$
E=-\sum_{i<j}\frac{m_im_j}{r_{ij}}
=-\left(\frac{3\cdot4}{5}+\frac{3\cdot5}{4}+\frac{4\cdot5}{3}\right)
\approx-12.82 .
$$

El sistema en conjunto está ligado. Para distinguir las etapas separamos esa
energía en dos partes. La **energía interna del par** $4$–$5$ usa su masa
reducida $\mu_{45}=m_4m_5/(m_4+m_5)$ y su posición y velocidad relativas:

$$
E_{45}=\frac12\,\mu_{45}\,|\mathbf v_4-\mathbf v_5|^2-\frac{m_4m_5}{|\mathbf r_4-\mathbf r_5|} .
$$

La **energía de la masa $3$ respecto del par** trata al par como una sola masa
$M_{45}=m_4+m_5$ ubicada en su centro de masa $\mathbf R_{45}$, que se mueve
con velocidad $\mathbf V_{45}$:

$$
E_3=\frac12\,\mu_3\,|\mathbf v_3-\mathbf V_{45}|^2-\frac{m_3M_{45}}{|\mathbf r_3-\mathbf R_{45}|},
\qquad \mu_3=\frac{m_3M_{45}}{m_3+M_{45}} .
$$

El par está ligado si $E_{45}<0$, y la masa $3$ se escapa si $E_3>0$. Con el
centro de masa total quieto, $E\approx E_{45}+E_3$. La aproximación es buena
cuando la masa $3$ está lejos del par, porque desde ahí el par se ve como una
masa puntual; cuando está cerca, deja de valer.

In [ ]:

def energias_jerarquicas(y, m, i, j):
    """Energía interna del par (i, j) y energía del tercer cuerpo respecto de su centro de masa."""
    k = 3 - i - j
    r, v = separar(y, 3)
    m_par = m[i] + m[j]
    e_par = 0.5 * m[i] * m[j] / m_par * np.sum((v[i] - v[j]) ** 2) - m[i] * m[j] / np.linalg.norm(r[i] - r[j])
    r_cm = (m[i] * r[i] + m[j] * r[j]) / m_par
    v_cm = (m[i] * v[i] + m[j] * v[j]) / m_par
    distancia = np.linalg.norm(r[k] - r_cm)
    e_tercero = 0.5 * m[k] * m_par / (m[k] + m_par) * np.sum((v[k] - v_cm) ** 2) - m[k] * m_par / distancia
    return e_par, e_tercero, distancia


t_pitagorico, delta_pitagorico = separaciones["pitagórico"]
y_pitagorico = trayectorias["pitagórico"][1]
print(f"{'t':>5} {'E_45':>8} {'E_3':>7} {'E_45 + E_3':>11} {'R masa 3':>9} {'δ':>9}")
for t_k in (36, 40, 44, 50, 54, 58, 62, 66, 70):
    k = np.argmin(np.abs(t_pitagorico - t_k))
    e_par, e_tercero, distancia = energias_jerarquicas(y_pitagorico[:, k], m_burrau, 1, 2)
    print(f"{t_k:5.0f} {e_par:8.2f} {e_tercero:7.2f} {e_par + e_tercero:11.2f} "
          f"{distancia:9.2f} {delta_pitagorico[k]:9.1e}")

Mientras los tres cuerpos interactúan, las dos energías cambian en cada
encuentro. Cuando son constantes, el sistema se comporta como dos problemas de
Kepler: el binario y la masa $3$ alrededor de él.

La energía total es negativa y, sin embargo, la masa $3$ escapa. No hay
contradicción: en el último encuentro el binario se **endurece**, $E_{45}$ se
vuelve más negativa, y la masa $3$ se lleva la energía liberada. Con dos
cuerpos y $E<0$ ninguno puede escapar; con tres, sí.

El crecimiento exponencial viene de la interacción entre los tres cuerpos, no
de la separación en un binario y un tercer cuerpo: un binario con un cuerpo
lejano se comporta casi como Kepler. Es lo que se veía en la animación de la
sección 3: durante más de cincuenta unidades de tiempo los círculos negros
acompañan a los cuerpos de color, y después se separan.

Los tres sistemas cumplen las mismas ecuaciones y se integran con el mismo
código y la misma tolerancia. La diferencia es física.

### 4.2. El exponente de Lyapunov

Cuando la separación crece en forma exponencial,

$$
\delta(t)\approx\delta_0\,e^{\lambda t},
$$

la tasa $\lambda$ es el **exponente de Lyapunov**. Formalmente,

$$
\lambda=\lim_{t\to\infty}\ \lim_{\delta_0\to0}\ \frac1t\ln\frac{\delta(t)}{\delta_0}.
$$

El orden de los límites importa: primero la perturbación infinitesimal, después
el tiempo largo. En una simulación ninguno de los dos se puede tomar. Por eso
estimamos una **tasa de crecimiento de tiempo finito**: ajustamos una recta a
$\ln\delta(t)$ en una ventana posterior a los saltos iniciales y anterior a la
formación del binario. La pendiente es nuestro estimador.

In [ ]:

def lambda_finito(t, delta, t_inicial, t_final):
    """Pendiente de ln(delta) en una ventana: un exponente de tiempo finito."""
    ventana = (t >= t_inicial) & (t <= t_final)
    return np.polyfit(t[ventana], np.log(delta[ventana]), 1)[0]


ventana_lambda = (33.0, 47.0)
lambda_pitagorico = lambda_finito(t_pitagorico, delta_pitagorico, *ventana_lambda)
print(f"λ finito en {ventana_lambda[0]:.0f} ≤ t ≤ {ventana_lambda[1]:.0f}: {lambda_pitagorico:.2f}")

La ventana elegida es parte de la afirmación: si incluye los saltos iniciales o
la meseta del binario, la pendiente deja de describir un crecimiento
exponencial.
Cambiarla un poco es un control de sensibilidad, no un detalle que se pueda
ocultar. Como el sistema pitagórico se desarma, este número no es un exponente
de Lyapunov asintótico del sistema: es un **exponente de tiempo finito** de
esta trayectoria y de esta ventana.

En un sistema ligado se usa el método de **Benettin** para no llegar a la
saturación. Se integran dos trayectorias durante un intervalo corto, se mide
la separación $\delta_k^-$, se acumula $\ln(\delta_k^-/\delta_0)$ y se vuelve
a poner la segunda trayectoria a distancia $\delta_0$ de la primera, en la
misma dirección del espacio de fases. Tras muchos intervalos,

$$
\lambda_M=\frac{\sum_{k=1}^{M}\ln(\delta_k^-/\delta_0)}
                 {\sum_{k=1}^{M}\Delta t_k}.
$$

La renormalización permite medir muchos tramos de crecimiento sin que la
separación se vuelva macroscópica. En un cálculo cuidadoso también se proyecta
la perturbación para respetar las cantidades conservadas. No lo aplicamos al
pitagórico porque, tras la eyección, ya no hay una dinámica ligada que recorrer
una y otra vez.

### 4.3. El horizonte de predicción

Si toleramos un error $\Delta$, la trayectoria calculada sirve hasta

$$
T_p\approx\frac1\lambda\ln\frac{\Delta}{\delta_0}.
$$

**Antes de ejecutar:** si dividimos $\delta_0$ por cien, ¿cuánto se alarga el
horizonte?

Si $\lambda$ fuera constante, la predicción sería un aumento de
$\ln(100)/\lambda$. Compárenla con las mediciones: los encuentros hacen que el
crecimiento real no sea uniforme.

Medimos $T_p$ como el primer instante en que $\delta>\Delta=0.1$. Como
indicador de lo que pasa al final, anotamos además qué cuerpo está más lejos del
centro de masa en $t=70$. La resolución temporal de esa medición es $0.05$.

In [ ]:

def horizonte(t, delta, tolerancia=0.1):
    """Primer instante en que la separación supera la tolerancia; nan si no la supera."""
    afuera = np.nonzero(delta > tolerancia)[0]
    return t[afuera[0]] if afuera.size else np.nan


def mas_lejano(y):
    r, _ = separar(y, 3)
    distancia = np.linalg.norm(r, axis=1)
    return m_burrau[np.argmax(distancia)], distancia.max()


t = np.linspace(0, t_final, 1401)
referencia = integrar(y_burrau, m_burrau, t_final, rtol=1e-13, t_eval=t)
masa, distancia = mas_lejano(referencia.y[:, -1])
print(f"referencia: más lejano en t = 70, m = {masa:g} a {distancia:.1f}")
print(f"{'δ0':>8} {'T_p':>6}  más lejano en t = 70")
for delta0 in (1e-4, 1e-6, 1e-8, 1e-10, 1e-12):
    y_b = y_burrau.copy()
    y_b[0] += delta0
    b = integrar(y_b, m_burrau, t_final, rtol=1e-13, t_eval=t)
    masa, distancia = mas_lejano(b.y[:, -1])
    delta = np.linalg.norm(b.y - referencia.y, axis=0)
    print(f"{delta0:8.0e} {horizonte(t, delta):6.1f}  m = {masa:g} a {distancia:.1f}")

Ocho órdenes de magnitud menos en $\delta_0$ no llegan a duplicar el horizonte.
El crecimiento no es uniforme: como la separación aumenta de a saltos en los
encuentros cercanos, el horizonte también avanza de a saltos. Con las
perturbaciones grandes cambia además el desenlace.

**Para pensar.** Entre $\delta_0=10^{-10}$ y $10^{-12}$ el horizonte deja de
crecer. ¿Qué otra perturbación, además de $\delta_0$, está separando las dos
trayectorias?

### 4.4. El error numérico es una perturbación más

`rtol` no es un error por paso de magnitud conocida: es la tolerancia relativa
que usa DOP853 para aceptar o rechazar un paso a partir de su estimación de
error local. La diferencia entre dos integraciones con tolerancias distintas
sí nos da una perturbación numérica medible; para la dinámica caótica, su
efecto se vuelve indistinguible del de perturbar la condición inicial.

In [ ]:

print(f"{'rtol':>8} {'T_p':>6} {'evaluaciones':>13}  más lejano en t = 70")
for rtol in (1e-6, 1e-8, 1e-10, 1e-12):
    sol = integrar(y_burrau, m_burrau, t_final, rtol=rtol, t_eval=t)
    masa, distancia = mas_lejano(sol.y[:, -1])
    delta = np.linalg.norm(sol.y - referencia.y, axis=0)
    print(f"{rtol:8.0e} {horizonte(t, delta):6.1f} {sol.nfev:13d}  m = {masa:g} a {distancia:.1f}")

Comparado con la referencia de `rtol = 1e-13`:

- antes del horizonte, bajar `rtol` acerca la trayectoria a la referencia: el
  estudio de convergencia de la clase 02 vale;
- después del horizonte, dos tolerancias dan **trayectorias distintas** y
  el estudio punto a punto deja de identificar una trayectoria única;
- cada factor cien en `rtol` alarga el horizonte, pero cuesta más evaluaciones
  y el horizonte crece sólo como un logaritmo.

En algunos sistemas caóticos se puede demostrar que la trayectoria numérica
sigue de cerca a alguna trayectoria exacta con una condición inicial vecina: se
dice que la **sombrea**. No es la trayectoria que pedimos, pero es una del
sistema.

### 4.5. Qué sigue siendo confiable

Las dos tablas sugieren qué afirmaciones resisten:

- la **energía**, el momento lineal y el angular cambian poco con todas las
  tolerancias razonables, antes y después del horizonte;
- el **desenlace**, quién se escapa y qué pareja queda ligada, se repite para
  perturbaciones chicas, pero no para cualquiera;
- la **trayectoria detallada** después del horizonte no se puede afirmar.

Las afirmaciones confiables pasan a ser **estadísticas**: qué fracción de un
conjunto de condiciones iniciales vecinas termina en cada desenlace, o cómo se
distribuye el tiempo de escape. Es el cambio de mirada que retoma la clase 05.

## 5. N cuerpos, en versión corta

### 5.1. El costo de sumar todas las parejas

La función `aceleraciones` de la sección 1 ya sirve para cualquier $N$ y en
tres dimensiones. Calcula $N(N-1)$ términos.

**Antes de ejecutar:** si duplicamos $N$, ¿por cuánto se multiplica el tiempo?

In [ ]:

rng = np.random.default_rng(2026)
tamanos = np.array([125, 250, 500, 1000, 2000])
tiempos = []
for n in tamanos:
    r = rng.normal(size=(n, 3))
    m = np.full(n, 1 / n)
    aceleraciones(r, m, suavizado=0.01)
    inicio = time.perf_counter()
    for _ in range(3):
        aceleraciones(r, m, suavizado=0.01)
    tiempos.append((time.perf_counter() - inicio) / 3)
tiempos = np.array(tiempos)
for n, segundos in zip(tamanos, tiempos):
    print(f"N = {n:5d}: {1e3 * segundos:6.1f} ms por evaluación")

In [ ]:

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(tamanos, 1e3 * tiempos, "o-", label="medido")
ax.loglog(tamanos, 1e3 * tiempos[-1] * (tamanos / tamanos[-1]) ** 2, "k--", lw=1, label="$\\propto N^2$")
ax.set(xlabel="$N$", ylabel="ms por evaluación")
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()

Con $N=2000$ también la memoria cuenta: el arreglo de diferencias tiene
$2000\times2000\times3$ números. Una galaxia tiene del orden de $10^{11}$
estrellas.

### 5.2. Suavizado

En un sistema de muchos cuerpos que representa una galaxia, cada partícula
simulada representa millones de estrellas: los encuentros cercanos entre dos
partículas no son físicos. Se los elimina con el **suavizado** que ya tiene la
función,

$$
\frac{1}{r}\ \longrightarrow\ \frac{1}{\sqrt{r^2+\varepsilon^2}} .
$$

Permite un paso fijo, pero tiene un precio: las escalas menores que
$\varepsilon$ dejan de estar resueltas, y la energía que se conserva es la del
potencial suavizado.

### 5.3. Cómo no sumar todas las parejas

Hay varias maneras de evitar las $N^2$ parejas. Cada una aprovecha algo del
problema, como una escala, una simetría o una jerarquía de masas, y paga con
una aproximación. La tabla resume las seis que describimos a continuación.

| Método | Costo | Idea | Qué se pierde |
|---|---|---|---|
| Suma directa | $N^2$ | todas las parejas | nada, salvo el suavizado |
| Corte y celdas | $N$ | sólo las vecinas a menos de un radio de corte | todas las interacciones lejanas |
| Partículas de prueba | $N\,M$ | sólo $M$ masas grandes generan gravedad; las otras $N$ partículas la sienten, pero no la producen | la atracción entre las $N$ partículas y su efecto sobre las masas grandes |
| Cáscaras esféricas | $N\log N$ | por el teorema de la cáscara esférica, la fuerza depende sólo de la masa interior | todo lo que no tiene simetría esférica |
| Malla (*particle-mesh*) | $N+M\log M$ | la masa se reparte en una grilla y Poisson se resuelve con transformada de Fourier | las escalas menores que una celda |
| Árbol de Barnes–Hut | $N\log N$ | un grupo lejano se reemplaza por su centro de masa | un error controlado por un ángulo de apertura |

Los esquemas que siguen son en el plano, para que se vean; los métodos son los
mismos en el espacio. Las muestras de partículas son sólo ilustrativas.

In [ ]:

rng_esquemas = np.random.default_rng(4)

# Partículas uniformes en el cuadrado unidad; la marcada es la más cercana al centro.
puntos = rng_esquemas.random((300, 2))
i_marcada = np.argmin(np.linalg.norm(puntos - 0.5, axis=1))

# Dos masas grandes, cada una con un disco de partículas de prueba.
masas_grandes = np.array([[0.3, 0.42], [0.72, 0.62]])
angulos = rng_esquemas.uniform(0, 2 * np.pi, 160)
radios_disco = rng_esquemas.uniform(0.04, 0.16, 160)
prueba = masas_grandes[np.arange(160) % 2] + radios_disco[:, None] * np.column_stack([np.cos(angulos), np.sin(angulos)])

# Esfera uniforme de radio 1 en tres dimensiones.
esfera = rng_esquemas.normal(size=(3000, 3))
esfera *= (rng_esquemas.random(3000) ** (1 / 3) / np.linalg.norm(esfera, axis=1))[:, None]

# Dos cúmulos dentro del cuadrado unidad, para el árbol.
cumulos = np.concatenate([rng_esquemas.normal([0.3, 0.35], 0.08, (150, 2)),
                          rng_esquemas.normal([0.7, 0.7], 0.12, (150, 2))])
cumulos = cumulos[np.all((cumulos >= 0) & (cumulos < 1), axis=1)]

#### Suma directa

$$
\mathbf a_i=\sum_{j\neq i}m_j\,
\frac{\mathbf r_j-\mathbf r_i}{\left(|\mathbf r_j-\mathbf r_i|^2+\varepsilon^2\right)^{3/2}} .
$$

Es la función `aceleraciones` de la sección 1: cada partícula interactúa con
las otras $N-1$. No aproxima nada, salvo el suavizado, y su costo es el de la
@fig-costo.

In [ ]:

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for p in puntos:
    ax.plot(*np.transpose([puntos[i_marcada], p]), color="0.75", lw=0.3)
ax.plot(*puntos.T, ".", color="0.4", ms=3)
ax.plot(*puntos[i_marcada], "o", color="C3", ms=8)
ax.set(aspect="equal", xticks=[], yticks=[], xlim=(0, 1), ylim=(0, 1))
plt.show()

#### Corte y celdas

Sólo se suman las parejas a menos de un **radio de corte** $r_c$:

$$
\mathbf a_i\approx\sum_{|\mathbf r_j-\mathbf r_i|<r_c}\mathbf f(\mathbf r_j-\mathbf r_i) .
$$

Para encontrarlas sin revisar todas las parejas, el espacio se divide en
**celdas** de lado $\ge r_c$: las vecinas de una partícula están en su celda o
en las adyacentes, nueve en el plano y veintisiete en el espacio. A densidad
fija, el número de candidatas no depende de $N$ y el costo es proporcional a
$N$. La aproximación sirve si la fuerza cae lo bastante rápido como para
despreciar lo que queda afuera del corte.

In [ ]:

lado_celda = 1 / 7
r_c = lado_celda
celda = np.floor(puntos / lado_celda).astype(int)
candidatas = np.all(np.abs(celda - celda[i_marcada]) <= 1, axis=1)
en_corte = np.linalg.norm(puntos - puntos[i_marcada], axis=1) < r_c
candidatas[i_marcada] = en_corte[i_marcada] = False
print(f"candidatas en las nueve celdas: {candidatas.sum()} de {len(puntos) - 1}")
print(f"a menos de r_c: {en_corte.sum()}")

In [ ]:

fig, ax = plt.subplots(figsize=(4.5, 4.5))
cx, cy = celda[i_marcada]
ax.add_patch(plt.Rectangle(((cx - 1) * lado_celda, (cy - 1) * lado_celda), 3 * lado_celda, 3 * lado_celda,
                           color="C0", alpha=0.12))
for x in np.arange(0, 1 + 1e-9, lado_celda):
    ax.axvline(x, color="0.8", lw=0.6)
    ax.axhline(x, color="0.8", lw=0.6)
ax.add_patch(plt.Circle(puntos[i_marcada], r_c, fill=False, color="C3", ls="--"))
ax.plot(*puntos[~candidatas].T, ".", color="0.6", ms=3)
ax.plot(*puntos[candidatas & ~en_corte].T, ".", color="C0", ms=5)
for p in puntos[en_corte]:
    ax.plot(*np.transpose([puntos[i_marcada], p]), color="C3", lw=0.6)
ax.plot(*puntos[en_corte].T, "o", color="C3", ms=3)
ax.plot(*puntos[i_marcada], "o", color="C3", ms=8)
ax.set(aspect="equal", xticks=[], yticks=[], xlim=(0, 1), ylim=(0, 1))
plt.show()

La clase 05 implementa este método para la fuerza de Lennard-Jones.

#### Partículas de prueba

Si unas pocas masas $M_k$ dominan el campo, se puede ignorar la gravedad de
todas las demás. Esas son **partículas de prueba**: sienten a las masas grandes,
pero no se atraen entre sí ni las mueven,

$$
\ddot{\mathbf r}_i=\sum_{k=1}^{M}M_k\,\frac{\mathbf R_k-\mathbf r_i}{|\mathbf R_k-\mathbf r_i|^3} .
$$

Las $M$ masas grandes se integran entre sí con la suma directa, y cada una de
las $N$ partículas de prueba suma $M$ términos. Así estudiaron
[Toomre y Toomre](https://doi.org/10.1086/151823) en 1972 los puentes y las colas
de galaxias en interacción: dos masas puntuales, cada una rodeada por un disco
de partículas de prueba. Se pierde la gravedad propia de los discos.

In [ ]:

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(*prueba.T, ".", color="0.5", ms=3)
ax.plot(*masas_grandes.T, "o", color="k", ms=12)
marcada = prueba[np.argmin(np.abs(radios_disco - 0.1) + (np.arange(160) % 2))]  # en el primer disco
for R in masas_grandes:
    ax.annotate("", xy=marcada + 0.85 * (R - marcada), xytext=marcada,
                arrowprops=dict(arrowstyle="->", color="C3", lw=1.2))
ax.plot(*marcada, "o", color="C3", ms=7)
ax.set(aspect="equal", xticks=[], yticks=[], xlim=(0.05, 0.95), ylim=(0.1, 0.9))
plt.show()

#### Cáscaras esféricas

Por el **teorema de la cáscara esférica** (*shell theorem*), que Newton
demostró en los *Principia* y que hoy sale en una línea de la ley de Gauss
para la gravedad, una cáscara esférica uniforme no ejerce fuerza
sobre lo que está adentro, y atrae a lo que está afuera como si toda su masa
estuviera en el centro. Si la distribución tiene simetría esférica alrededor
del origen, cada partícula siente sólo la masa más cerca del centro que ella:

$$
\mathbf a_i=-\frac{M(<r_i)}{r_i^2}\,\hat{\mathbf r}_i,
\qquad
M(<r_i)=\sum_{j:\ r_j<r_i}m_j .
$$

El nombre no implica elegir un espesor: cada partícula hace de cáscara delgada
de su propio radio, y $M(<r_i)$ cuenta las partículas interiores una por una.
Para tener $M(<r)$ de todas las partículas alcanza con ordenarlas por radio: de
ahí el costo $N\log N$. Una variante agrupa las partículas en cáscaras de
espesor fijo, y entonces ese espesor es un parámetro más del método. El
resultado es exacto con simetría esférica. Si la
distribución no es esférica, el método la trata como si lo fuera.

In [ ]:

radio = np.linalg.norm(esfera, axis=1)
corte_z = np.abs(esfera[:, 2]) < 0.1  # una rebanada delgada, para la figura
i_capa = np.flatnonzero(corte_z)[np.argmin(np.abs(radio[corte_z] - 0.6))]
interiores = radio < radio[i_capa]
print(f"r_i = {radio[i_capa]:.2f}: {interiores.sum()} de {len(esfera)} partículas son interiores")

In [ ]:

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(*esfera[corte_z & ~interiores, :2].T, ".", color="0.7", ms=3)
ax.plot(*esfera[corte_z & interiores, :2].T, ".", color="C0", ms=3)
ax.add_patch(plt.Circle((0, 0), radio[i_capa], fill=False, color="C3", ls="--"))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="0.4"))
p = esfera[i_capa, :2]
ax.annotate("", xy=0.25 * p, xytext=p, arrowprops=dict(arrowstyle="->", color="C3", lw=1.5))
ax.plot(*p, "o", color="C3", ms=8)
ax.plot(0, 0, "+", color="k", ms=10)
ax.set(aspect="equal", xticks=[], yticks=[], xlim=(-1.1, 1.1), ylim=(-1.1, 1.1))
plt.show()

#### Malla

El método de malla (*particle-mesh*) calcula la gravedad en una grilla de $M$
celdas de lado $h$, en cuatro pasos:

1. **Asignación.** La masa de las partículas se reparte en la grilla. Con
   **nube en celda** (*cloud in cell*), cada partícula es un cuadrado uniforme
   del tamaño de una celda, un cubo en el espacio, y su masa se reparte entre
   las celdas que toca en proporción al área compartida: cuatro celdas en el
   plano y ocho en el espacio.
2. **Potencial.** Con la masa $m_c$ de cada celda se resuelve la ecuación de
   Poisson, $\nabla^2\phi=4\pi\rho$. Es lo mismo que convolucionar la masa con
   $-1/r$,
   $$
   \phi_c=-\sum_{c'}\frac{m_{c'}}{|\mathbf x_c-\mathbf x_{c'}|},
   $$
   y la transformada rápida de Fourier hace esa convolución con un costo
   $M\log M$. La transformada supone un sistema periódico: para una
   distribución aislada se duplica la grilla en cada dirección y se rellena con
   ceros, y así la convolución periódica coincide con la del sistema aislado.
3. **Fuerza.** $\mathbf a=-\nabla\phi$, con diferencias centradas en la
   grilla.
4. **Interpolación.** La aceleración se lleva de la grilla a cada partícula con
   los mismos pesos de la asignación. Usar los mismos pesos evita que una
   partícula se empuje a sí misma.

Se pierde todo lo que es menor que una celda: el tamaño $h$ hace el papel del
suavizado. La referencia clásica es el libro de
[Hockney y Eastwood](https://doi.org/10.1201/9781439822050).

Los pesos de la nube en celda de una partícula, con las posiciones en unidades
de $h$ y los centros de las celdas en los semienteros:

In [ ]:

particula = np.array([1.35, 1.70])
inferior = np.floor(particula - 0.5).astype(int)  # celda cuyo centro queda abajo a la izquierda
fx, fy = particula - 0.5 - inferior
pesos = {(0, 0): (1 - fx) * (1 - fy), (1, 0): fx * (1 - fy), (0, 1): (1 - fx) * fy, (1, 1): fx * fy}
for (dx, dy), peso in pesos.items():
    print(f"celda ({inferior[0] + dx}, {inferior[1] + dy}): {peso:.3f}")
print(f"suma: {sum(pesos.values()):.3f}")

In [ ]:

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for x in range(4):
    ax.axvline(x, color="0.7", lw=0.8)
    ax.axhline(x, color="0.7", lw=0.8)
izquierda, abajo = particula - 0.5
for (dx, dy), peso in pesos.items():
    x0, y0 = inferior[0] + dx, inferior[1] + dy
    x_a, x_b = max(izquierda, x0), min(izquierda + 1, x0 + 1)
    y_a, y_b = max(abajo, y0), min(abajo + 1, y0 + 1)
    ax.add_patch(plt.Rectangle((x_a, y_a), x_b - x_a, y_b - y_a, color=f"C{dx + 2 * dy}", alpha=0.35))
    ax.plot(x0 + 0.5, y0 + 0.5, "+", color=f"C{dx + 2 * dy}", ms=10)
    ax.text(x0 + 0.5, y0 + 0.3, f"{peso:.2f}", ha="center", va="top", color=f"C{dx + 2 * dy}")
ax.add_patch(plt.Rectangle((izquierda, abajo), 1, 1, fill=False, color="k", lw=1.2))
ax.plot(*particula, "o", color="k", ms=6)
ax.set(aspect="equal", xticks=range(4), yticks=range(4), xlim=(0, 3), ylim=(0, 3),
       xlabel="$x/h$", ylabel="$y/h$")
plt.show()

#### Árbol de Barnes–Hut

Visto desde lejos, un grupo de partículas se parece a una sola masa puntual
ubicada en su centro de masa. El error de esa aproximación depende del cociente
entre el tamaño del grupo, $s$, y su distancia, $d$: es del orden de
$(s/d)^2$, porque el término siguiente del desarrollo en multipolos, el dipolo,
se anula alrededor del centro de masa. El árbol de
[Barnes y Hut](https://doi.org/10.1038/324446a0) (1986) organiza las partículas
en grupos de todos los tamaños, para usar en cada caso el grupo más grande que
está lo bastante lejos.

**Construcción.** La raíz es un cuadrado que contiene todas las partículas. Si
tiene más de una, se divide en cuatro cuadrantes, ocho octantes en el espacio,
y se repite con cada cuadrante que no esté vacío hasta que cada hoja tiene una
sola partícula. Cada nodo guarda su lado, su masa total y su centro de masa.

**Recorrido.** Para calcular la fuerza sobre una partícula se empieza por la
raíz y, en cada nodo:

- si es una hoja, se suma la fuerza de su partícula, salvo que sea la propia;
- si $s/d<\theta$, con $d$ la distancia de la partícula al centro de masa del
  nodo, se suma la fuerza de toda su masa ubicada en su centro de masa;
- si no, se **abre** el nodo y se repite con cada uno de sus hijos.

El **ángulo de apertura** $\theta$ controla el error: con $\theta=0$ se abre
todo y se recupera la suma directa.

In [ ]:

def construir_arbol(r, m, indices, centro, lado, nodos):
    """Agrega a `nodos` el cuadrado de centro y lado dados, con las partículas `indices`, y sus hijos.

    Cada nodo guarda su centro, su lado, su masa, su centro de masa, sus
    partículas y los índices de sus hijos. Devuelve el índice del nodo agregado.
    """
    k = len(nodos)
    cm = r[indices[0]] if len(indices) == 1 else m[indices] @ r[indices] / m[indices].sum()
    nodos.append({"centro": centro, "lado": lado, "masa": m[indices].sum(), "cm": cm,
                  "particulas": indices, "hijos": []})
    if len(indices) > 1:
        for esquina in ((-1, -1), (-1, 1), (1, -1), (1, 1)):
            c = centro + np.array(esquina) * lado / 4
            adentro = indices[np.all((r[indices] >= c - lado / 4) & (r[indices] < c + lado / 4), axis=1)]
            if adentro.size:
                nodos[k]["hijos"].append(construir_arbol(r, m, adentro, c, lado / 2, nodos))
    return k


def nodos_vistos(nodos, r_i, theta, k=0, registro=None):
    """Índices de los nodos que usa una partícula en r_i.

    Si se pasa una lista `registro`, le agrega (nodo, s/d, decisión) por cada
    nodo visitado, con decisión "hoja", "propia", "usar" o "abrir".
    """
    nodo = nodos[k]
    d = np.linalg.norm(nodo["cm"] - r_i)
    if not nodo["hijos"]:
        decision = "propia" if d == 0 else "hoja"
    else:
        decision = "usar" if nodo["lado"] < theta * d else "abrir"
    if registro is not None:
        registro.append((k, nodo["lado"] / d if d > 0 else np.inf, decision))
    if decision == "abrir":
        return [j for hijo in nodo["hijos"] for j in nodos_vistos(nodos, r_i, theta, hijo, registro)]
    return [] if decision == "propia" else [k]


def aceleracion_arbol(nodos, r, theta):
    """Aceleración de cada partícula sumando sólo los nodos que ve, como masas puntuales."""
    a = np.zeros_like(r)
    for i, r_i in enumerate(r):
        for k in nodos_vistos(nodos, r_i, theta):
            d = nodos[k]["cm"] - r_i
            a[i] += nodos[k]["masa"] * d / np.linalg.norm(d) ** 3
    return a

Un ejemplo con ocho partículas de igual masa. Tres están abajo a la izquierda,
cerca de la partícula $0$, cuya fuerza vamos a calcular; una está abajo a la
derecha, y cuatro forman un grupo arriba a la derecha.

In [ ]:

r_chico = np.array([[0.08, 0.10], [0.35, 0.30], [0.15, 0.40], [0.60, 0.20],
                    [0.70, 0.70], [0.80, 0.85], [0.90, 0.65], [0.62, 0.90]])
m_chico = np.full(len(r_chico), 1 / len(r_chico))
nodos_chico = []
construir_arbol(r_chico, m_chico, np.arange(len(r_chico)), np.array([0.5, 0.5]), 1.0, nodos_chico)

# Letras para los nodos internos; las hojas se nombran por su partícula.
internos = [k for k, nodo in enumerate(nodos_chico) if nodo["hijos"]]
nombre = {k: "ABCDEFGH"[internos.index(k)] if k in internos else f"partícula {nodos_chico[k]['particulas'][0]}"
          for k in range(len(nodos_chico))}

theta_chico = 0.7
recorrido = []
vistos_chico = nodos_vistos(nodos_chico, r_chico[0], theta_chico, registro=recorrido)
print(f"Recorrido para la partícula 0 con θ = {theta_chico}:")
print(f"{'nodo':>12} {'partículas':>12} {'s/d':>6}  decisión")
for k, s_sobre_d, decision in recorrido:
    particulas = " ".join(str(i) for i in nodos_chico[k]["particulas"])
    cociente = f"{s_sobre_d:6.2f}" if decision in ("usar", "abrir") else f"{'—':>6}"  # en las hojas no se usa
    print(f"{nombre[k]:>12} {particulas:>12} {cociente}  {decision}")
for theta in (0.5, theta_chico):
    print(f"θ = {theta}: {len(nodos_vistos(nodos_chico, r_chico[0], theta))} interacciones, "
          f"de {len(r_chico) - 1} con la suma directa")

In [ ]:

decisiones = {k: decision for k, _, decision in recorrido}
colores = {"abrir": "0.8", "usar": "C0", "hoja": "C3", "propia": "white"}

fig, (ax_plano, ax_grafo) = plt.subplots(1, 2, figsize=(10, 4.6))
for k, nodo in enumerate(nodos_chico):
    esquina = nodo["centro"] - nodo["lado"] / 2
    if decisiones.get(k) == "usar":
        ax_plano.add_patch(plt.Rectangle(esquina, nodo["lado"], nodo["lado"], color="C0", alpha=0.15))
        ax_plano.plot(*nodo["cm"], "+", color="C0", ms=12, mew=2)
    ax_plano.add_patch(plt.Rectangle(esquina, nodo["lado"], nodo["lado"], fill=False, color="0.5", lw=0.8))
    if nodo["hijos"]:
        ax_plano.text(*(esquina + [0.012, nodo["lado"] - 0.012]), nombre[k], va="top", fontsize=11, weight="bold")
for k in vistos_chico:
    ax_plano.plot(*np.transpose([r_chico[0], nodos_chico[k]["cm"]]), color=colores[decisiones[k]], lw=1)

# Criterio para el nodo usado: s/d < θ equivale a que su centro de masa quede a más de s/θ.
nodo_usado = nodos_chico[next(k for k in vistos_chico if decisiones[k] == "usar")]
s_usado = nodo_usado["lado"]
arcos = []
for theta, color in ((0.5, "C2"), (0.7, "C1")):
    ax_plano.add_patch(plt.Circle(r_chico[0], s_usado / theta, fill=False, color=color, ls="--", lw=1.5))
    arcos.append(plt.Line2D([], [], color=color, ls="--", lw=1.5, label=f"radio $s/\\theta$, $\\theta$ = {theta}"))
x_izquierda = nodo_usado["centro"][0] - s_usado / 2
y_abajo = nodo_usado["centro"][1] - s_usado / 2
ax_plano.annotate("", xy=(x_izquierda, y_abajo - 0.03), xytext=(x_izquierda + s_usado, y_abajo - 0.03),
                  arrowprops=dict(arrowstyle="<->", color="C0", lw=1.5))
ax_plano.text(x_izquierda + s_usado / 2, y_abajo - 0.045, "$s$", color="C0", ha="center", va="top", fontsize=13)
medio = (r_chico[0] + nodo_usado["cm"]) / 2
ax_plano.text(medio[0] - 0.035, medio[1] + 0.02, "$d$", color="C0", ha="right", fontsize=13)
ax_plano.legend(handles=arcos, loc="lower right", fontsize=8)
for i, p in enumerate(r_chico):
    ax_plano.plot(*p, "o", color="C3" if i == 0 else "k", ms=7 if i == 0 else 5)
    ax_plano.text(p[0] + 0.015, p[1] + 0.015, str(i), fontsize=9)
ax_plano.set(aspect="equal", xticks=[], yticks=[], xlim=(0, 1), ylim=(0, 1))

# Posición de cada nodo en el grafo: las hojas en orden, cada nodo interno sobre sus hijos.
posicion = {}


def ubicar(k, nivel, contador):
    hijos = nodos_chico[k]["hijos"]
    if hijos:
        for hijo in hijos:
            ubicar(hijo, nivel + 1, contador)
        posicion[k] = (np.mean([posicion[h][0] for h in hijos]), -nivel)
    else:
        posicion[k] = (contador[0], -nivel)
        contador[0] += 1


ubicar(0, 0, [0])
for k, nodo in enumerate(nodos_chico):
    for hijo in nodo["hijos"]:
        ax_grafo.plot(*np.transpose([posicion[k], posicion[hijo]]), color="0.6", lw=1, zorder=1)
for k, nodo in enumerate(nodos_chico):
    texto = nombre[k] if nodo["hijos"] else str(nodo["particulas"][0])
    ax_grafo.scatter(*posicion[k], s=650, color=colores.get(decisiones.get(k), "white"),
                     edgecolor="k" if k in decisiones else "0.7", zorder=2)
    ax_grafo.text(*posicion[k], texto, ha="center", va="center", fontsize=10,
                  weight="bold" if nodo["hijos"] else "normal", zorder=3)
ax_grafo.set(xticks=[], yticks=[], xlim=(-0.7, len(r_chico) - 0.3), ylim=(min(y for _, y in posicion.values()) - 0.6, 0.6))
for borde in ax_grafo.spines.values():
    borde.set_visible(False)
plt.show()

La partícula $0$ hace cuatro interacciones en lugar de siete: el grupo de
arriba a la derecha cuenta como una sola masa. El criterio $s/d<\theta$ es lo
mismo que $d>s/\theta$: el nodo se usa si su centro de masa está más lejos que
el arco de radio $s/\theta$. Con $\theta=0.7$ el arco queda antes de la cruz;
con $\theta=0.5$ queda después, el nodo se abre, porque $s/d\approx0.52$, y
vuelven a ser siete interacciones. Con ocho partículas la
ganancia es chica. Con muchas, cada partícula termina interactuando con un
número de nodos que crece como $\log N$: de ahí el costo $N\log N$. Con
dos cúmulos de unas 300 partículas:

In [ ]:

m_cumulos = np.full(len(cumulos), 1 / len(cumulos))
nodos = []
construir_arbol(cumulos, m_cumulos, np.arange(len(cumulos)), np.array([0.5, 0.5]), 1.0, nodos)
a_directa = aceleraciones(cumulos, m_cumulos)
print(f"{len(cumulos)} partículas, {len(nodos)} nodos")
for theta in (0.0, 0.5, 1.0):
    vistos = np.mean([len(nodos_vistos(nodos, r_i, theta)) for r_i in cumulos])
    error = np.linalg.norm(aceleracion_arbol(nodos, cumulos, theta) - a_directa, axis=1) / np.linalg.norm(a_directa, axis=1)
    print(f"θ = {theta:.1f}: {vistos:5.1f} nodos por partícula, error relativo mediano {np.median(error):.0e}")

In [ ]:

i_arbol = np.argmin(np.linalg.norm(cumulos - [0.3, 0.35], axis=1))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for nodo in nodos:
    ax.add_patch(plt.Rectangle(nodo["centro"] - nodo["lado"] / 2, nodo["lado"], nodo["lado"],
                               fill=False, color="0.8", lw=0.5))
ax.plot(*cumulos.T, ".", color="0.5", ms=3)
for k in nodos_vistos(nodos, cumulos[i_arbol], 0.5):
    nodo = nodos[k]
    if nodo["hijos"]:
        ax.add_patch(plt.Rectangle(nodo["centro"] - nodo["lado"] / 2, nodo["lado"], nodo["lado"],
                                   color="C0", alpha=0.15))
        ax.plot(*nodo["cm"], "+", color="C0", ms=7)
    ax.plot(*np.transpose([cumulos[i_arbol], nodo["cm"]]), color="C3", lw=0.4)
ax.plot(*cumulos[i_arbol], "o", color="C3", ms=8)
ax.set(aspect="equal", xticks=[], yticks=[], xlim=(0, 1), ylim=(0, 1))
plt.show()

#### En la práctica

Los códigos de producción combinan varios: malla para las distancias grandes y
árbol o suma directa para las cortas. Ninguno es gratis: cada uno cambia la
física de una manera que hay que conocer. El corte con celdas, que veremos en la
clase 05, es exacto para fuerzas de corto alcance.

**Para pensar.** ¿Por qué el corte no sirve para la gravedad? Piensen cuántas
partículas hay en una cáscara de radio $r$ y cuánto aporta cada una.

## 6. Qué podemos concluir y qué no

Podemos concluir que:

- una solución especial, como la órbita en ocho, sirve de test aunque no haya
  solución general, con un límite de resolución que fijan sus propios datos;
- en un encuentro cercano el paso adaptativo cambia en órdenes de magnitud;
- la misma ecuación da crecimiento lineal, lento o exponencial de las
  perturbaciones según el sistema;
- el horizonte de predicción crece sólo como el logaritmo de la precisión;
- después del horizonte, el error numérico y la incertidumbre en la condición
  inicial son indistinguibles;
- sumar todas las parejas cuesta $N^2$, y cada alternativa cambia algo de la
  física.

No podemos concluir que:

- conservar la energía garantice que la trayectoria sea la correcta;
- bajar `rtol` haga converger la trayectoria a cualquier tiempo;
- un exponente de Lyapunov sea un número único, sin decir cómo se estimó;
- una animación de muchos cuerpos convincente sea una simulación confiable.

### Puente con el TP 04

Al final de la clase se publica el [TP 04](../TPs/04-colapso/index.md). Van a
simular el colapso de una esfera fría con suma directa, cáscaras esféricas y
malla, y a decidir qué observable reproduce cada aproximación.

### Pregunta de salida

La pregunta de salida:

> **¿Qué parte de una simulación de tres cuerpos podemos afirmar después del
> horizonte de predicción, y con qué evidencia?**

## Epílogo: tres soles

El problema de los tres cuerpos también tiene su literatura. En la trilogía de
ciencia ficción de **Liu Cixin**, *El problema de los tres cuerpos*, *El bosque
oscuro* y *El fin de la muerte*, publicada en China entre 2008 y 2010, una
civilización vive en un planeta que gira alrededor de tres estrellas. Su
historia está hecha de la dinámica de esta clase: hay **eras estables**, en las
que el planeta orbita una sola de las estrellas y el clima se puede prever, y
**eras caóticas**, en las que las tres se disputan el planeta y no hay
calendario posible. Durante siglos intentan predecir el movimiento de sus soles,
y fracasan una y otra vez: una civilización entera viviendo más allá del
horizonte de predicción. La primera novela ganó el premio Hugo en 2015; fue la
primera vez que lo ganaba un autor asiático.

![La trilogía de Liu Cixin en la edición en castellano de Nova.](https://cursos.colavecchia.net/fisica-computacional/contenidos/clases/figuras/04-tres-cuerpos-tapas.jpg)

Las tapas de la edición en castellano acompañan el recorrido de la trilogía. En
la primera, una pirámide en una llanura helada, bajo soles que no guardan
ningún orden: el mundo visto desde adentro del caos. En la segunda, un planeta
solo frente a una estructura oscura: el título alude a la hipótesis del
**bosque oscuro**, según la cual el universo calla porque cada civilización se
esconde de las demás como un cazador en un bosque de noche. En la tercera, una
figura cae hacia un círculo negro: la escala pasa de un sistema estelar al
destino del universo.

La novela tuvo dos adaptaciones. En China, **Three-Body** (三体), de Tencent y la
televisión estatal CCTV, se estrenó en enero de 2023: treinta episodios
dirigidos por Yang Lei, muy fieles al primer libro. En marzo de 2024 Netflix
estrenó
[*El problema de los 3 cuerpos*](https://www.netflix.com/title/81024821), de
David Benioff y D. B. Weiss, los creadores de *Game of Thrones*, y Alexander
Woo: lleva buena parte de la historia a Inglaterra, condensa el primer libro y
adelanta elementos de los otros dos.

### Alfa Centauri, el sistema de verdad

En la novela, los tres soles son los de **Alfa Centauri**, el sistema estelar
más cercano al Sol, a unos 4.3 años luz. El sistema real no es caótico: es
**jerárquico**. Alfa Centauri A y B forman un binario, con un período de unos
80 años, y la tercera estrella, Próxima Centauri, gira alrededor del par a unas
13 000 unidades astronómicas, con un período de más de medio millón de años.
Vista desde Próxima, la pareja A–B es casi una sola masa puntual. Es la
configuración de un binario y un cuerpo lejano que apareció en las etapas del
problema pitagórico, cuando la masa 3 se alejaba sin escapar, y en ella la
dinámica es casi la de dos problemas de Kepler. Próxima tiene al menos un
planeta, Próxima b, descubierto en 2016
([Anglada-Escudé y colaboradores](https://doi.org/10.1038/nature19106)).

### GW Orionis: tres estrellas que desgarran un disco

Un sistema triple con consecuencias que se pueden fotografiar es **GW
Orionis**, a unos 1300 años luz, en la constelación de Orión. Durante once
años, un equipo siguió las órbitas de sus tres estrellas con los instrumentos
AMBER y GRAVITY del interferómetro del VLT, de ESO: medir un sistema triple es,
otra vez, un problema de órbitas. Encontraron que las tres órbitas no están en
un mismo plano, ni entre sí ni con el disco de gas y polvo que rodea al sistema
([Kraus y colaboradores](https://doi.org/10.1126/science.aba4633); ver también
el [comunicado de ESO](https://www.eso.org/public/news/eso2014/)).

Las imágenes del instrumento SPHERE, en el VLT, y del radiotelescopio ALMA
muestran el resultado: el disco no es plano, está deformado y partido en
anillos. El anillo interior, con unas 30 masas terrestres de polvo, está
desalineado del resto, y las simulaciones del equipo muestran que la
desalineación de las órbitas puede desgarrar el disco de esa manera. Si en ese
anillo se forman planetas, tendrán órbitas muy inclinadas y tres soles en el
cielo.

![Izquierda: impresión artística del anillo interior de GW Orionis, reconstruido a partir de las observaciones. Derecha: la imagen de SPHERE, donde se ve la sombra del anillo sobre el resto del disco. Crédito: ESO/L. Calçada, Exeter/Kraus et al., licencia CC BY 4.0.](https://cursos.colavecchia.net/fisica-computacional/contenidos/clases/figuras/04-gw-orionis-anillo.jpg)

![El disco de GW Orionis visto por ALMA (izquierda), con sus anillos separados, y por SPHERE (derecha). Crédito: ALMA (ESO/NAOJ/NRAO), ESO/Exeter/Kraus et al., licencia CC BY 4.0.](https://cursos.colavecchia.net/fisica-computacional/contenidos/clases/figuras/04-gw-orionis-alma-sphere.jpg)

## Referencias

**Órbita en ocho**

- C. Moore, "Braids in classical dynamics", *Physical Review Letters* **70**,
  3675–3679 (1993). <https://doi.org/10.1103/PhysRevLett.70.3675>
- A. Chenciner y R. Montgomery, "A remarkable periodic solution of the
  three-body problem in the case of equal masses", *Annals of Mathematics*
  **152**, 881–901 (2000). <https://doi.org/10.2307/2661357>
- C. Simó, "New families of solutions in N-body problems", en *European
  Congress of Mathematics*, Progress in Mathematics **201**, 101–115,
  Birkhäuser (2001). <https://doi.org/10.1007/978-3-0348-8268-2_6>

**Problema pitagórico**

- C. Burrau, "Numerische Berechnung eines Spezialfalles des
  Dreikörperproblems", *Astronomische Nachrichten* **195**, 113–118 (1913).
  <https://doi.org/10.1002/asna.19131950602>
- V. Szebehely y C. F. Peters, "Complete solution of a general problem of
  three bodies", *The Astronomical Journal* **72**, 876 (1967).
  <https://doi.org/10.1086/110355>

**Regularización**

- K. F. Sundman, "Mémoire sur le problème des trois corps", *Acta
  Mathematica* **36**, 105–179 (1913). <https://doi.org/10.1007/BF02422379>
- T. Levi-Civita, "Sur la régularisation du problème des trois corps", *Acta
  Mathematica* **42**, 99–144 (1920). <https://doi.org/10.1007/BF02404404>
- P. Kustaanheimo y E. Stiefel, "Perturbation theory of Kepler motion based on
  spinor regularization", *Journal für die reine und angewandte Mathematik*
  **218**, 204–219 (1965). <https://doi.org/10.1515/crll.1965.218.204>

**Métodos para muchos cuerpos**

- A. Toomre y J. Toomre, "Galactic bridges and tails", *The Astrophysical
  Journal* **178**, 623 (1972). <https://doi.org/10.1086/151823>
- J. Barnes y P. Hut, "A hierarchical O(N log N) force-calculation
  algorithm", *Nature* **324**, 446–449 (1986).
  <https://doi.org/10.1038/324446a0>
- R. W. Hockney y J. W. Eastwood, *Computer Simulation Using Particles*,
  Adam Hilger (1988). <https://doi.org/10.1201/9781439822050>

**Ficción y observaciones**

- Liu Cixin, *El problema de los tres cuerpos*, *El bosque oscuro* y *El fin
  de la muerte*, Nova. Trilogía publicada en chino entre 2008 y 2010.
- G. Anglada-Escudé y colaboradores, "A terrestrial planet candidate in a
  temperate orbit around Proxima Centauri", *Nature* **536**, 437–440 (2016).
  <https://doi.org/10.1038/nature19106>
- S. Kraus y colaboradores, "A triple-star system with a misaligned and warped
  circumstellar disk shaped by disk tearing", *Science* **369**, 1233–1238
  (2020). <https://doi.org/10.1126/science.aba4633>
- ESO, "New observations show planet-forming disc torn apart by its three
  central stars", comunicado eso2014 (2020).
  <https://www.eso.org/public/news/eso2014/>